# Module 4: Q&A RAG Pipeline

This notebook implements the Retrieval-Augmented Generation (RAG) pipeline for the e-commerce customer support chatbot. It includes:
- Loading and exploring the dataset
- Creating knowledge base chunks
- Generating embeddings
- Building a FAISS index
- Testing retrieval and generation

In [ ]:
# Disable tqdm progress bars to prevent ipykernel ContextVar errors
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
import datasets
datasets.disable_progress_bar()

import os
import pickle
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from groq import Groq

import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.config import (
    FAISS_INDEX_PATH,
    KB_METADATA_PATH,
    EMBEDDING_MODEL_NAME,
    GROQ_API_KEY,
    GROQ_MODEL,
    RAG_TOP_K,
    RAG_SIMILARITY_THRESHOLD,
    RAG_SYSTEM_PROMPT,
    RAG_USER_PROMPT,
    LANGUAGE_NAMES
)

## 1. Load Dataset and EDA

We load the `bitext/Bitext-customer-support-llm-chatbot-training-dataset` from Hugging Face and explore it.

In [ ]:
# Load dataset
print("Loading dataset from Hugging Face...")
dataset = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')
df = dataset['train'].to_pandas()

# Show dataset info and sample
print(f"Dataset shape: {df.shape}")
display(df[['instruction', 'response', 'intent', 'category']].sample(5))

# Calculate response lengths
df['response_length'] = df['response'].apply(lambda x: len(str(x).split()))

# EDA Plots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Response length distribution
sns.histplot(data=df, x='response_length', bins=50, ax=axes[0])
axes[0].set_title('Distribution of Response Lengths (words)')
axes[0].set_xlabel('Words')

# Plot 2: Category distribution
category_counts = df['category'].value_counts()
sns.barplot(y=category_counts.index, x=category_counts.values, ax=axes[1])
axes[1].set_title('Category Distribution')
axes[1].set_xlabel('Count')
plt.tight_layout()
plt.show()

## 2. Knowledge Base and Embeddings

We prepare the knowledge base chunks and embed the 'instruction' column (the user questions) using `sentence-transformers/all-MiniLM-L6-v2`. This allows us to find the most similar pre-answered question for a user's query.

In [ ]:
# Build knowledge base chunks
kb_metadata = df[['instruction', 'response', 'intent', 'category']].to_dict('records')

# Load Embedding Model
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Generate embeddings for 'instruction'
print("Generating embeddings... This may take a few minutes.")
instructions = [item['instruction'] for item in kb_metadata]
embeddings = embedder.encode(instructions, show_progress_bar=False)
print(f"Embeddings shape: {embeddings.shape}")

# Show sample similarity
q1_idx = 0
q2_idx = 1
q3_idx = len(instructions) - 1

print(f"Q1: {instructions[q1_idx]}")
print(f"Q2: {instructions[q2_idx]}")
print(f"Q3: {instructions[q3_idx]}")

from sklearn.metrics.pairwise import cosine_similarity
sim_1_2 = cosine_similarity([embeddings[q1_idx]], [embeddings[q2_idx]])[0][0]
sim_1_3 = cosine_similarity([embeddings[q1_idx]], [embeddings[q3_idx]])[0][0]

print(f"\nSimilarity between Q1 and Q2: {sim_1_2:.4f}")
print(f"Similarity between Q1 and Q3: {sim_1_3:.4f}")

## 3. Building FAISS Index

We use `IndexFlatIP` from FAISS, which performs maximum inner product search. By normalizing the L2 norm of the embeddings beforehand, inner product becomes equivalent to cosine similarity. This is computationally efficient and works well for this scale (~26k records).

In [ ]:
# Normalize embeddings for cosine similarity using inner product
faiss.normalize_L2(embeddings)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"FAISS index built with {index.ntotal} vectors.")

# Save index and metadata
os.makedirs(str(FAISS_INDEX_PATH.parent), exist_ok=True)
faiss.write_index(index, str(FAISS_INDEX_PATH))

os.makedirs(str(KB_METADATA_PATH.parent), exist_ok=True)
with open(KB_METADATA_PATH, 'wb') as f:
    pickle.dump(kb_metadata, f)
    
print(f"Index saved to: {FAISS_INDEX_PATH}")
print(f"Metadata saved to: {KB_METADATA_PATH}")

## 4. Test Retrieval

Let's test the retrieval step independently. Given a query, we should find the top-3 most similar questions in the KB.

In [ ]:
test_query = "How do I cancel my order?"

# Embed query
query_embedding = embedder.encode([test_query])
faiss.normalize_L2(query_embedding)

# Retrieve top 3
k = 3
scores, indices = index.search(query_embedding, k)

print(f"Query: {test_query}\n")
for j, i in enumerate(indices[0]):
    score = scores[0][j]
    match = kb_metadata[i]
    print(f"Match {j+1} (Score: {score:.4f})")
    print(f"Instruction: {match['instruction']}")
    print(f"Response: {match['response']}")
    print(f"Category: {match['category']}\n")

## 5. Test Generation with LLM (Groq)

Finally, let's use the Groq API to combine the retrieved context and generate a final response, applying proper formatting based on language and sentiment.

In [ ]:
if not GROQ_API_KEY:
    print("WARNING: GROQ_API_KEY is not set. The generation step will not work.")
    print("Testing retrieval only:")
    # We already tested retrieval above.
else:
    client = Groq(api_key=GROQ_API_KEY)
    
    test_queries = [
        "How can I return a damaged item?",
        "Do you ship internationally?",
        "Where is my refund?"
    ]
    
    for query in test_queries:
        print(f"--- Query: {query} ---")
        
        # 1. Retrieve
        q_emb = embedder.encode([query])
        faiss.normalize_L2(q_emb)
        scores, indices = index.search(q_emb, RAG_TOP_K)
        
        retrieved_chunks = []
        for j, i in enumerate(indices[0]):
            if scores[0][j] >= RAG_SIMILARITY_THRESHOLD:
                retrieved_chunks.append(kb_metadata[i])
                
        # 2. Format Context
        context_str = "\n\n".join([
            f"Question: {chunk['instruction']}\nAnswer: {chunk['response']}"
            for chunk in retrieved_chunks
        ])
        
        # 3. Format Prompt
        sentiment = 'neutral'
        language = 'en'
        lang_name = LANGUAGE_NAMES.get(language, 'English')
        
        system_prompt = RAG_SYSTEM_PROMPT.format(detected_language_name=lang_name, detected_sentiment=sentiment)
        user_prompt = RAG_USER_PROMPT.format(user_message=query, context=context_str)
        
        # 4. Generate
        try:
            chat_completion = client.chat.completions.create(
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                model=GROQ_MODEL,
                temperature=0.3,
                max_tokens=512,
            )
            response = chat_completion.choices[0].message.content
            print("Response:")
            print(response)
        except Exception as e:
            print(f"Error generating response: {e}")
        print("\n")

## 6. Architecture and Design Choices

### RAG Architecture
The Retrieval-Augmented Generation (RAG) architecture used here operates in two main phases:
1. **Retrieval**: When a user asks a question, we convert it into an embedding. We then search our FAISS index to find the most similar questions in our training dataset. We retrieve the corresponding verified answers (context).
2. **Generation**: We combine the user's question, the retrieved context, and system instructions (including detected language and sentiment) into a prompt. We pass this to a Large Language Model (LLM) via the Groq API to generate a final, coherent, and context-aware response.

### Embedding Choice
We use `sentence-transformers/all-MiniLM-L6-v2`. This model is chosen because:
- **Efficiency**: It is small, fast, and generates 384-dimensional embeddings, which is perfect for real-time applications and fits easily in memory.
- **Performance**: Despite its small size, it performs exceptionally well on semantic similarity tasks, mapping similar intents to similar points in the vector space.

### FAISS IndexFlatIP
We use `IndexFlatIP` (Inner Product) from FAISS after L2-normalizing our embeddings. 
- **Cosine Similarity**: Inner product of L2-normalized vectors is mathematically equivalent to cosine similarity, which is standard for measuring text similarity.
- **Scale**: At ~26,000 records, an exact search (`IndexFlat`) is extremely fast. We don't need approximate nearest neighbor (ANN) techniques like HNSW or IVF, which add complexity and index-building overhead. `IndexFlatIP` guarantees finding the exact nearest neighbors almost instantaneously for this dataset size.